# Reference RDMs vs the curated hierarchy

The directory tree under `images/<variant>/` **is** the Kiani-Mur semantic hierarchy:
`animate/animal/body/bird/duck3.png` encodes domain, then mid-level, then basic level.

`D_sem_km` is built from that tree, so it should reproduce it *exactly* (section 2 is a
hard check, not a correlation). Every other RDM is an independent measurement, and how
well it agrees with the tree is an open question:

| RDM | what it measures | derived from the tree? |
|---|---|---|
| `sem_km` | tree path length | yes, by construction |
| `sem_wn` | WordNet shortest path between synsets | no |
| `clip_pre` / `clip_post` | CLIP ViT-B/32 embedding cosine | no |
| `sens_pre` / `sens_post` | raw pixel Euclidean | no |

**Motivating question.** The hierarchy is a hand-curated classification. If the
NN-derived (CLIP) and lexical (WordNet) RDMs turn out to live on a completely different
semantic surface, that matters for RQ3, where `D_sem` enters the decomposition as a
predictor.

**Run from the repo root** so `analysis.*` imports resolve, with the RDMs already built
(`python -m analysis.rdms.build_all`).

In [1]:
import numpy as np
import pandas as pd

from analysis.rdms import hierarchy_comparison as hc
from analysis.rdms.common import load_rdm

RDMS = hc.load_rdms()
list(RDMS)

['sem_km', 'sem_wn', 'clip_pre', 'clip_post', 'sens_pre', 'sens_post']

## 1. The shape of the tree

Before comparing anything to the hierarchy, look at the hierarchy itself. In particular,
check whether the leaves sit at a single depth: if they do not, the tree is *ragged* and
KM distance stops being a pure function of how far up two images diverge.

In [2]:
parts = hc.path_parts()
depths = hc.path_depths(parts)

print(f"images: {len(parts)}")
print("leaf depth distribution:")
print(pd.Series(depths.astype(int)).value_counts().sort_index().to_string())
print(f"\nragged tree: {hc.depth_is_ragged(parts)}")

for level, name in hc.LEVEL_NAMES.items():
    n_cat = len({p[:level] for p in parts if len(p) >= level})
    print(f"  level {level} ({name}): {n_cat} categories")

images: 725
leaf depth distribution:
3    336
4    316
5     73

ragged tree: True
  level 1 (domain (animate / inanimate)): 2 categories
  level 2 (mid-level category): 4 categories
  level 3 (basic level): 16 categories


**Why raggedness matters.** With leaves at several depths, two images in a finely
subdivided branch (say `animate/animal/body/monkey/baboon/`) are pushed further apart in
KM than two equally related images in a coarse branch, purely because their branch has
more levels. KM distance is therefore *not* comparable across branches, and any
level-stratified analysis (RQ1b, RQ2d) inherits that asymmetry. Worth stating explicitly
in the write-up rather than discovering later.

## 2. Sanity check: `sem_km` is the tree, exactly

`d(i,j) = depth_i + depth_j - 2 * lca_depth(i,j)`, floored at 1 off-diagonal.

`reconstruct_km_from_hierarchy()` reimplements the LCA computation independently of
`semantic_km.build_km_rdm()`, on purpose: if both called the same helper this check would
be a tautology. Exact array equality is the bar here, not a correlation.

In [3]:
km_rebuilt = hc.reconstruct_km_from_hierarchy()
km_stored = RDMS["sem_km"]

assert np.array_equal(km_rebuilt, km_stored), "sem_km does not match the directory tree"
print(f"sem_km reproduces the tree exactly ({len(km_stored):,} pairs, max |diff| = "
      f"{np.abs(km_rebuilt - km_stored).max()})")

lca = hc.lca_depth_condensed(parts)
print("\nKM distance is not a function of LCA depth alone (ragged tree):")
print(pd.crosstab(pd.Series(lca, name="lca_depth"),
                  pd.Series(km_stored.astype(int), name="km_distance")).to_string())

sem_km reproduces the tree exactly (262,450 pairs, max |diff| = 0.0)

KM distance is not a function of LCA depth alone (ragged tree):
km_distance     1      2      3      4      5      6      7      8     9
lca_depth                                                               
0               0      0      0      0      0      0  75600  45003  6643
1               0      0      0  21983  19317  11424  11972      0     0
2               0  27596  11059   6834   2493   1290      0      0     0
3            6901   7269   1960      0      0      0      0      0     0
4            3768    990      0      0      0      0      0      0     0
5             348      0      0      0      0      0      0      0     0


Each LCA depth maps to several KM distances. That spread is the raggedness from
section 1, made concrete.

## 3. Question one: what shares rank structure?

Pairwise Spearman between every RDM. This asks only whether two RDMs order pairs the
same way; it says nothing about the hierarchy's *levels*, which is section 4's job.
The two questions get separate figures deliberately: a high correlation does not imply
a clean depth gradient, and vice versa.

In [4]:
corr = hc.correlation_matrix(rdms=RDMS)
hc.plot_correlation_matrix(corr).show()

In [5]:
print("agreement with the curated hierarchy, ranked:")
print(corr["sem_km"].drop("sem_km").sort_values(ascending=False).round(3).to_string())

print(f"\nCLIP vs WordNet: {corr.loc['clip_pre', 'sem_wn']:.3f}")

agreement with the curated hierarchy, ranked:
clip_pre     0.348
clip_post    0.310
sem_wn       0.286
sens_pre     0.090
sens_post    0.054

CLIP vs WordNet: 0.047


## 4. Question two: does distance grow with hierarchical separation?

Mean distance at each LCA depth, normalised to each RDM's value at the root so that
pixel Euclidean and cosine sit on one scale. A monotonically falling curve means the RDM
places hierarchically closer images closer together. `sem_km` is dashed: it is the
reference, and its curve is true by construction.

In [6]:
profile = hc.depth_profile(rdms=RDMS)
hc.plot_depth_profile(profile).show()

In [7]:
print(profile.round(3).to_string())

print("\nroot / deepest ratio (higher = steeper hierarchical gradient):")
for col in profile.columns.drop("n_pairs"):
    print(f"  {col:10s} {profile[col].iloc[0] / profile[col].iloc[-1]:6.3f}"
          f"   monotonic: {bool(np.all(np.diff(profile[col]) <= 0))}")

           n_pairs  sem_km  sem_wn  clip_pre  clip_post  sens_pre  sens_post
lca_depth                                                                   
0           127246   1.000   1.000     1.000      1.000     1.000      1.000
1            64696   0.698   1.012     0.912      0.928     0.979      1.005
2            49272   0.370   0.810     0.853      0.869     0.984      1.001
3            16130   0.227   0.568     0.628      0.665     0.905      0.911
4             4758   0.162   0.262     0.479      0.509     0.835      0.852
5              348   0.134   0.559     0.353      0.368     0.885      0.907

root / deepest ratio (higher = steeper hierarchical gradient):
  sem_km      7.458   monotonic: True
  sem_wn      1.789   monotonic: False
  clip_pre    2.829   monotonic: True
  clip_post   2.716   monotonic: True
  sens_pre    1.130   monotonic: False
  sens_post   1.102   monotonic: False


Note the deepest bin holds few pairs, so treat its endpoint cautiously. Any
non-monotonicity there is worth chasing down to specific categories rather than reading
as a global property, which is what section 6 does for WordNet.

## 5. Is the agreement real, and is it semantic?

Two things to rule out.

**Chance.** A condensed RDM has 262,450 entries from 725 images, so the entries are
massively non-independent and any analytic p-value is meaningless. The right null
permutes *image identity*, shuffling rows and columns together.

**Low-level confounding.** CLIP could track the hierarchy only because hierarchically
related images happen to look alike at the pixel level. Partialling out `sens_pre`
separates those.

In [8]:
# n_perm=2000 for a reportable result; drop to ~300 for a quick pass.
N_PERM = 2000

for name in ["sem_wn", "clip_pre", "sens_pre"]:
    res = hc.label_shuffle_test(RDMS["sem_km"], RDMS[name], n_perm=N_PERM)
    print(f"  sem_km vs {name:9s}  {res}")
print(f"\n(p floor is 1/{N_PERM + 1} = {1 / (N_PERM + 1):.5f})")

  sem_km vs sem_wn     rho=+0.286  null=-0.0001+/-0.0146  z=19.6  p=0.00050


  sem_km vs clip_pre   rho=+0.348  null=-0.0001+/-0.0127  z=27.5  p=0.00050


  sem_km vs sens_pre   rho=+0.090  null=-0.0006+/-0.0141  z=6.4  p=0.00050

(p floor is 1/2001 = 0.00050)


In [9]:
raw = corr.loc["sem_km", "clip_pre"]
ctrl = hc.partial_correlation(RDMS["sem_km"], RDMS["clip_pre"], RDMS["sens_pre"])
print(f"rho(km, clip)            = {raw:+.3f}")
print(f"rho(km, clip | pixels)   = {ctrl:+.3f}")
print(f"rho(clip, wn | km)       = "
      f"{hc.partial_correlation(RDMS['clip_pre'], RDMS['sem_wn'], RDMS['sem_km']):+.3f}")

rho(km, clip)            = +0.348
rho(km, clip | pixels)   = +0.338


rho(clip, wn | km)       = -0.059


## 6. Are CLIP and WordNet the same semantic surface?

If they were two noisy readings of one construct, their joint prediction of the hierarchy
would be much less than the sum of the parts. If the joint R^2 lands near that sum, they
are carrying largely non-overlapping information, and `D_sem = WN` is a *different
specification* rather than a robustness check on `D_sem = KM`.

In [10]:
print(hc.incremental_r2(
    RDMS["sem_km"],
    {"clip_pre": RDMS["clip_pre"], "sem_wn": RDMS["sem_wn"]},
).round(4).to_string(index=False))

          predictors     r2
            clip_pre 0.1212
              sem_wn 0.0816
   clip_pre + sem_wn 0.1940
(sum if independent) 0.2029


In [11]:
# Chase the WordNet depth-profile reversal down to specific categories.
from collections import Counter

from analysis.rdms.common import load_manifest

deepest = lca == lca.max()
syn = load_manifest()["wn_synset_name"].to_numpy()
iu = np.triu_indices(len(parts), 1)
pairs = [tuple(sorted((syn[i], syn[j])))
         for i, j in zip(iu[0][deepest], iu[1][deepest])]

print(f"synset pairs among the {int(deepest.sum())} deepest-LCA pairs:")
for pair, n in Counter(pairs).most_common(8):
    mask = np.array([p == pair for p in pairs])
    print(f"  {pair[0]:16s} x {pair[1]:16s} n={n:4d}  "
          f"mean WN d = {RDMS['sem_wn'][deepest][mask].mean():6.2f}")

synset pairs among the 348 deepest-LCA pairs:
  baboon.n.01      x chimpanzee.n.01  n=  24  mean WN d =   7.00
  baboon.n.01      x macaque.n.01     n=  18  mean WN d =   2.00
  baboon.n.01      x baboon.n.01      n=  15  mean WN d =   0.00
  baboon.n.01      x capuchin.n.01    n=  12  mean WN d =  18.00
  baboon.n.01      x orangutan.n.01   n=  12  mean WN d =   7.00
  chimpanzee.n.01  x macaque.n.01     n=  12  mean WN d =   7.00
  goat.n.01        x sheep.n.01       n=   8  mean WN d =   2.00
  capuchin.n.01    x chimpanzee.n.01  n=   8  mean WN d =  19.00


## 7. Level by level

Cohen's d between between-category and within-category distances at each level of the
tree. Positive d means the RDM recovers that level; larger means it recovers it more
sharply. This is the level-stratified view that RQ1b will need.

In [12]:
compare = ["sem_wn", "clip_pre", "clip_post", "sens_pre"]

rows = {}
for level, label in hc.LEVEL_NAMES.items():
    rows[f"L{level}: {label}"] = hc.level_separation(
        level, rdms={k: RDMS[k] for k in compare}
    )["cohens_d"]

pd.DataFrame(rows).T.round(3)

rdm,sem_wn,clip_pre,clip_post,sens_pre
L1: domain (animate / inanimate),0.444,0.804,0.717,0.172
L2: mid-level category,0.850,0.958,0.893,0.195
L3: basic level,1.278,1.795,1.668,0.534


## 8. Takeaways

Fill in against the numbers above; as of the run on the corrected manifest with
`ViT-B-32-quickgelu` CLIP:

1. **`sem_km` reproduces the tree exactly.** The reference RDM is sound, and the tree is
   ragged (leaves at depths 3, 4 and 5), so KM distances are not comparable across
   branches.
2. **CLIP tracks the curated hierarchy better than WordNet does** (rho 0.35 vs 0.29),
   with the steepest and only cleanly monotonic depth gradient of the non-KM RDMs, and it
   survives partialling out pixels. So the NN embedder is *not* on a foreign semantic
   surface.
3. **CLIP and WordNet are near-orthogonal to each other** (rho 0.05, slightly negative
   once KM is held fixed) while each independently tracks KM. Their joint R^2 on KM is
   close to the sum of the individual ones, i.e. complementary, not redundant.
4. **WordNet's gradient reverses at the deepest level**, driven by primate categories:
   WordNet routes Old World to New World monkeys up through the primate root, so
   `baboon x capuchin` scores far higher than `baboon x macaque`. That is taxonomic
   bookkeeping, not perceptual or folk-semantic distance.
5. **Pixels are nearly flat across depth**, as expected, and serve as the floor.

Implications for the pre-reg are discussed alongside RQ3; the key one is that `D_sem`
under WordNet and under KM are different constructs rather than robustness arms of one.

None of this is confirmatory: it compares reference RDMs to each other and to the
hierarchy, with no human perceptual data involved. RQ1b needs the SpAM data.